In [ ]:
import pandas as pd

In [ ]:
df_train, df_test, df_val = pd.read_csv("train.csv"), pd.read_csv("test.csv"), pd.read_csv("validation.csv")

In [ ]:
### 判断训练集，测试集，验证集是否有重复值和缺失值，如有则进行处理

def check(data):
    if data.isna().any(axis=None):
        print("存在缺失值")
        data.dropna(subset=["english", "french"])
    
    if len(data["english"]) != len(set(data["english"])):
        print("存在重复值")
        data = data.drop_duplicates(subset=["english", "french"], keep="first", ignore_index=True)
    print("数据清洗完成")
    return data


df_train = check(df_train)
df_test = check(df_test)
df_val = check(df_val)


In [ ]:
def split_train_and_val(data):
    df_shuffled = data.sample(frac=1, random_state=42)
    n_valid = round(len(df_shuffled) * 0.2)
    valid_df = df_shuffled.iloc[: n_valid].copy().reset_index(drop=True)
    train_df = df_shuffled[n_valid: ].copy().reset_index(drop=True)
    shared = set(valid_df["english"]) & set(train_df["english"])
    assert len(shared) == 0, "训练集和验证集存在重复"

    print(f"训练集数据量{len(train_df)}")
    print(f"验证集数据量{len(valid_df)}")
    return train_df, valid_df

train_df, valid_df = split_train_and_val(df_train)

In [ ]:
from transformers import AutoTokenizer
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize_data(data):
    def f(prompt):
        txt = prompt.strip("###").strip("\n").strip()
        ls = txt.split("\n")
        tmp = []
        for t in ls:
            if t == '':
                continue
            tmp.append(t.strip("###"))
        return ''.join(tmp)
    inputs = [f(p) for p in data["prompt"].to_list()]
    tgt = data["french"].to_list()
    encoded = tokenizer(inputs)
    tgt_encoded = tokenizer(text_target=tgt)
    encoded["labels"] = tgt_encoded["input_ids"]
    return encoded

In [ ]:
train_tokens = tokenize_data(train_df)
valid_tokens = tokenize_data(valid_df)


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict(dict(train_tokens))
valid_dataset = Dataset.from_dict(dict(valid_tokens))


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.02,
    target_modules=["q", "v"],
    bias="none"
)

In [ ]:
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./lora_outputs",

    learning_rate=5e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=2,

    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,

    logging_steps=10,
    report_to="none",
    label_names=["labels"],
    seed=42,
)

In [ ]:
import numpy as np

references = valid_df["french"].tolist()

def compute_metrics(eval_pred):
    prediction_ids = eval_pred.predictions

    # Trainer 可能用 -100 补齐不同长度的预测，解码前换成 padding token
    prediction_ids = np.where(
        prediction_ids == -100,
        tokenizer.pad_token_id,
        prediction_ids,
    )

    predictions = tokenizer.batch_decode(
        prediction_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    correct = sum(
        prediction.strip() == reference.strip()
        for prediction, reference in zip(predictions, references)
    )

    return {"exact_match": correct / len(references)}


In [ ]:
model.generation_config.max_new_tokens = 64
model.generation_config.num_beams = 4
model.generation_config.do_sample = False

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("当前使用的设备：", trainer.args.device)

In [ ]:
trainer.train()
after = trainer.predict(
    valid_dataset,
    metric_key_prefix="eval",
)

print(after.metrics)

print("微调后完全匹配率：", after.metrics["eval_exact_match"])

In [ ]:
### eval
import torch 
trained_model = trainer.model
trained_model.eval()

def translate_sentences(sentences):
    def f(prompt):
        txt = prompt.strip("###").strip("\n").strip()
        ls = txt.split("\n")
        tmp = []
        for t in ls:
            if t == '':
                continue
            tmp.append(t.strip("###"))
        return ''.join(tmp)
    inputs = [f(p) for p in sentences]
    encoded = tokenizer(
        inputs,
        padding=True,
        return_tensors="pt"
    ).to(trained_model.device)

    with torch.inference_mode():
        output_ids = trained_model.generate(
            **encoded,
            max_new_tokens=64,
            num_beams=1,
            do_sample=False,
        )
        return tokenizer.batch_decode(
            output_ids,
            skip_special_tokens=True,
            clean_up_tokenization_space=False,
        )



In [ ]:
prompt = df_test["prompt"].iloc[0]

translation = translate_sentences([prompt])[0]

print("英语原句：", sentence)
print("模型翻译：", translation)

In [ ]:
from tqdm.auto import tqdm

prompts = df_test["prompt"].tolist()
test_predictions = []

batch_size = 8

for start in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[start : start + batch_size]

    batch_predictions = translate_sentences(batch_prompts)
    test_predictions.extend(batch_predictions)

assert len(test_predictions) == len(df_test)

translated_test_df = df_test[["english"]].copy()
translated_test_df["prediction"] = test_predictions

translated_test_df.head(5)